In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://bh.rdm.yzwlab.com/'
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
default_result_path = None
close_on_fail = False
transition_timeout = 60000

tljh_url = 'http://localhost'
tljh_username = 'admin'
tljh_password = 'change-your-password'

project_name = None

# Use Firefox for popup handling (Chromium has issues with popup events)
browser_type = 'firefox'


In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if tljh_username is None:
    tljh_username = input('TLJH username')
if tljh_password is None:
    tljh_password = getpass(prompt=f'Password for {tljh_username}@TLJH')
if project_name is None:
    project_name = datetime.now().strftime('TEST-BINDERHUB-%Y%m%d%H%M')

binderhub_url = tljh_url
project_url = None
project_created = False

# プロジェクトに対するBinderHubアドオンの登録

- サブシステム名: アドオン
- ページ/アドオン: BinderHub
- 機能分類: アカウント設定
- シナリオ名: プロジェクトへの有効化
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM, BinderHub, GRDMは全てプロフィールを埋めていること / JupyterHubはサーバーが5つ以内の状態であること)、TLJHのログイン情報


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


## TLJHのURLを開く

ログインページが表示されることを確認する

In [ ]:
import asyncio

async def _step(page):
    await page.goto(tljh_url)
    await expect(page.locator('#username_input')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)  # wait for animation

await run_pw(_step)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()
    await expect(page.locator('//button[text() = "ログイン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「GakuNin Federated Computing Services (Jupyter)」の「有効にする」をクリックする

アドオン利用規約の確認ダイアログが表示されること

In [ ]:
async def _step(page):
    addon_name = 'GakuNin Federated Computing Services (Jupyter)'
    enable_locator = page.locator(f'//div[@full_name = "{addon_name}"]//a[text() = "有効にする"]')
    if await enable_locator.count():
        await enable_locator.click()
        confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
        await expect(confirm_button).to_be_visible(timeout=transition_timeout)
        await confirm_button.click()
    else:
        print('Addon already enabled')

await run_pw(_step)


## 「BinderHubを追加」ボタンをクリックする

BinderHubクライアント情報の設定ダイアログが表示されること

In [ ]:
async def _step(page):
    binder_entry = page.locator(f'//a[text() = "{binderhub_url}"]')
    if await binder_entry.count():
        print('BinderHub entry already exists')
        return
    add_button = page.locator('//button[@href="#binderhubInputHost"]')
    await add_button.click()
    await expect(page.locator('//input[@name = "binderhub_url"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//input[@name = "binderhub_url"]').fill(binderhub_url)
    # Tab to trigger any on-change events
    await page.locator('//input[@name = "binderhub_url"]').press('Tab')
    save_button = page.locator('//button[contains(@data-bind, "hostCompleted") and contains(text(), "保存")]')
    await expect(save_button).to_be_enabled(timeout=transition_timeout)
    await save_button.click()
    await expect(page.locator(f'//a[text() = "{binderhub_url}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 追加したBinderHubのチェックボックスをクリックする

Default BinderHub URL がTLJHのURLになること

In [ ]:
async def _step(page):
    default_square = page.locator(f'//a[text() = "{binderhub_url}"]/../..//i[contains(@class, "fa-square")]')
    if await default_square.count():
        await default_square.click()
        await expect(page.locator(f'//a[text() = "{binderhub_url}"]/../..//i[contains(@class, "fa-check-square")]')).to_be_visible(timeout=transition_timeout)
    else:
        print('BinderHub already selected')

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

TLJHのログインページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

TLJHのログインページが表示されること

In [ ]:
async def _step(page):
    # 新しいタブを開く
    popup_future = page.wait_for_event('popup', timeout=transition_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    
    await expect(popup.locator('#username_input')).to_be_visible(timeout=transition_timeout)
    return popup

await run_pw(_step)


## TLJHのユーザー名・パスワードを入力し、Sign inをクリックする

TLJHのビルド画面が表示されること

In [ ]:
import traceback

async def _step(page):
    await page.locator('#username_input').fill(tljh_username)
    await page.locator('#password_input').fill(tljh_password)
    await page.locator('#login_submit').click()

    sync_icon = page.locator(f'//a[text()="{project_url}"]/../..//*[@data-testid="SyncIcon"]')
    try:
        await expect(sync_icon).to_be_visible(timeout=transition_timeout)
    except:
        traceback.print_exc()
        await page.reload()
        await expect(sync_icon).to_be_visible(timeout=transition_timeout)
    await sync_icon.click()
    await asyncio.sleep(3)

await run_pw(_step)

## Statusがチェックマークになるまで待つ

In [ ]:
import traceback

async def _step(page):
    await page.locator('//*[contains(text(), "Close")]').click()
    retry_count = 40
    while retry_count > 0:
        try:
            await expect(page.locator(f'//a[text()="{project_url}"]/../..//*[@data-testid="CheckIcon"]')).to_be_visible(timeout=transition_timeout)
            break
        except:
            retry_count -= 1
            if retry_count == 0:
                traceback.print_exc()
            await page.reload()

    try:
        await expect(page.locator(f'//a[text()="{project_url}"]/../..//*[@data-testid="CheckIcon"]')).to_be_visible(timeout=transition_timeout)
    except:
        try:
            await page.locator(f'//a[text()="{project_url}"]/../..//*[@data-testid="SyncIcon"]').click()
            await asyncio.sleep(30)
        except:
            traceback.print_exc()
        raise

await run_pw(_step)

## 「Servers」をクリックする

In [ ]:
async def _step(page):
    #<a href="/services/tljh_repo2docker/servers">Servers</a>
    await page.locator('//a[@href="/services/tljh_repo2docker/servers"]').click()
    # <button class="MuiButtonBase-root MuiButton-root MuiButton-contained MuiButton-containedPrimary MuiButton-sizeMedium MuiButton-containedSizeMedium MuiButton-colorPrimary MuiButton-root MuiButton-contained MuiButton-containedPrimary MuiButton-sizeMedium MuiButton-containedSizeMedium MuiButton-colorPrimary css-1xscybk" tabindex="0" type="button">Create new Server<span class="MuiTouchRipple-root css-w0pj6f"></span></button>
    await expect(page.locator('//button[contains(text(), "Create new Server")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「Create new Server」をクリックする

In [ ]:
async def _step(page):
    await page.locator('//button[contains(text(), "Create new Server")]').click()
    await expect(page.locator(f'//a[text()="{project_url}"]/../..//input[@type="checkbox"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## このプロジェクトURLに対応する環境のチェックボックスをクリックし、サーバー名に「test-server」と入力し、「Create Server」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//a[text()="{project_url}"]/../..//input[@type="checkbox"]').click()
    await page.locator('#server_name').fill('test-server')
    await page.locator('//button[contains(text(), "Create Server")]').click()
    await expect(page.locator('#server_name')).not_to_be_visible(timeout=transition_timeout * 5)

await run_pw(_step)

## 「test-server」の「Open Server」をクリックする


In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup')
    await page.locator('//a[contains(text(), "Open Server")]').click()
    popup = await popup_future

    #<div class="jp-LauncherCard" title="Python 3 (ipykernel)" tabindex="0" data-category="Notebook"><div class="jp-LauncherCard-icon"><img src="/user/admin/test-server/kernelspecs/python3/logo-svg.svg" class="jp-Launcher-kernelIcon" alt="Python 3 (ipykernel)"></div><div class="jp-LauncherCard-label" title="Python 3 (ipykernel)"><p>Python 3 (ipykernel)</p></div></div>
    await expect(popup.locator('//*[@class="jp-LauncherCard" and @title="Python 3 (ipykernel)" and @data-category="Notebook"]')).to_be_visible(timeout=transition_timeout)

    return popup

await run_pw(_step)

## JupyterLabウィンドウを閉じる

In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)


## 「Stop Server」をクリックし、「Accept」をクリックする

In [ ]:
async def _step(page):
    await page.locator('//*[contains(text(), "Stop Server")]').click()
    await page.locator('//*[contains(text(), "Accept")]').click()
    try:
        await expect(page.locator('//a[contains(text(), "Open Server")]')).not_to_be_visible(timeout=transition_timeout)
    except:
        traceback.print_exc()
        await page.reload()
        await expect(page.locator('//a[contains(text(), "Open Server")]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## TLJHのブラウザタブを閉じ、GRDMの解析ページに戻る

In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)

## 右上のユーザーアイコンをクリックする

ユーザーメニューが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-auth-dropdown]').click()
    await expect(page.locator('//*[@data-test-ad-settings]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「設定」をクリックする

ユーザー設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-ad-settings]').click()
    await expect(page.locator('//*[text() = "パーソナルアクセストークン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「パーソナルアクセストークン」をクリックする

トークン一覧が表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "パーソナルアクセストークン"]').click()
    token_table = page.locator('//a[contains(@href, "/settings/tokens/")]')
    await expect(token_table).not_to_have_count(0, timeout=transition_timeout)

await run_pw(_step)


## 「非アクティブ」列の「×」を全てクリックする

トークンが存在しない状態になること

In [ ]:
async def _step(page):
    while True:
        delete_icon = page.locator('//i[contains(@class, "fa-times") and contains(@class, "text-danger")]').first
        if not await delete_icon.count():
            break
        await delete_icon.click()
        confirm_button = page.locator('//*[@data-bb-handler = "confirm"]')
        await expect(confirm_button).to_be_visible(timeout=transition_timeout)
        await confirm_button.click()
        await expect(page.locator('//*[contains(@class, "alert-success")]')).to_be_visible(timeout=transition_timeout)
        await page.reload()
        await page.locator('//h3[text() = "パーソナルアクセストークン"]').click()
        await asyncio.sleep(5)
    message = page.locator('//p[text() = "GakuNin RDMに接続できるアクセストークンを作成していません。"]')
    await expect(message).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


終了処理を実施する。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}